# ESRS E1-2 — Climate-Related Physical Risk Assessment with CLIMADA

**Identification of Climate-Related Risks and Scenario Analysis**  
*Draft Simplified ESRS (EFRAG Technical Advice V1.5, 30 Nov 2025)*

This notebook runs the full CLIMADA **probabilistic** risk pipeline starting from nothing
but **site geolocations (WGS84 lat/lon)**. Everything else — the hazard event sets — comes
from **open-access data** served by the CLIMADA Data API (ETH Zurich):

| Hazard | Open-access source | CLIMADA `haz_type` |
|---|---|---|
| Flooding (river) | GloFAS / ISIMIP river-flood depth maps | `RF` |
| Extreme precipitation | Derived from the river-flood model (precip → flood → damage) | `RF` |
| Storms (tropical cyclone) | Synthetic tracks from IBTrACS, scaled by CMIP6 | `TC` |
| Wildfires | FIRMS-based wildfire event sets | `WF` |
| Landslides | NASA COOLR / global landslide susceptibility | `LS` |
| Heat Stress / Heatwaves | ISIMIP heat-stress exceedance datasets | `HS` |
| Drought & Water Stress | ISIMIP relative crop-yield loss (rainfed wheat proxy) | `RC` |
| Sea-Level Rise | ISIMIP coastal-flood / SLR inundation maps | `CF` |

The risk equation is **Risk = Hazard × Exposure × Vulnerability**. The output is the
**Expected Annual Loss (EAL)** per site — the core E1-2 metric — computed over a full
probabilistic event set (return-period flood maps / thousands of synthetic storms).

**Run matrix:** 8 hazards × 3 scenarios (SSP1-1.9, SSP2-4.5, SSP3-7.0) × 3 time horizons
= **72 runs per site**.

> Built and tested on **climada 6.1**. Uses `ImpactCalc(...).impact()` (the `Impact.calc()`
> method used in older guides is removed in climada 6.x).
>
> **Data availability note:** Wildfire, landslide, heat stress, drought, and sea-level rise
> datasets have more limited coverage in the CLIMADA open API than flood/TC. Combinations
> that return no data are recorded as `NaN` and flagged in the disclosure table — the
> framework still satisfies E1-2 by documenting the gap and the alternative data source
> needed to fill it.

## 1 · Imports

In [1]:
import pandas as pd
import numpy as np

from climada.entity import Exposures, ImpactFuncSet, ImpactFunc, ImpfTropCyclone
from climada.hazard import Hazard
from climada.engine import ImpactCalc
from climada.util.api_client import Client

## 2 · Load the exposure (your only input: geolocations)

Each row is one site. The **only data you need to supply are `latitude` / `longitude`**.

Because asset replacement values are company-specific (and often the one thing you don't
have yet), `value` defaults to **1.0** for every site. With unit value, the resulting EAL is
the **expected annual loss as a fraction of asset value** (i.e. the mean annual damage
ratio, 0–1). When you obtain real book/replacement values, drop them into the `value`
column and the same EAL becomes a currency figure — no other change needed.

The enrichment columns (`headcount`, `business_area`, `criticality`) are **not** used in the
CLIMADA math; they enrich the E1-2 *sensitivity* narrative (AR 6(b)) and carry through to the
final disclosure table. `country_iso3` is required to fetch the per-country open-access
hazard tiles. `impf_RF` / `impf_TC` link each site to its vulnerability curve (defined below).

In [ ]:
# --- The only mandatory inputs: site geolocations (WGS84 decimal degrees) ---
sites = pd.DataFrame({
    'site_name':     ['Manila WH', 'London HQ', 'Singapore DC', 'Paris Office'],
    'latitude':      [14.5995,     51.5074,     1.3521,         48.8566],
    'longitude':     [120.9842,   -0.1278,      103.8198,       2.3522],
    'country_iso3':  ['PHL',       'GBR',        'SGP',          'FRA'],

    # Asset value: default 1.0 -> EAL is expressed as a fraction of asset value.
    # Replace with book / replacement cost (same currency) to get monetary EAL.
    'value':         [1.0,         1.0,          1.0,            1.0],

    # --- Enrichment (post-processing only; not used in CLIMADA's math) ---
    'headcount':     [120,         450,          85,             200],
    'business_area': ['Logistics', 'Corporate HQ','Data Center',  'Sales'],
    'criticality':   ['High',      'Critical',   'Critical',     'Medium'],

    # --- Vulnerability links: which impact-function id applies per hazard type ---
    'impf_RF':       [1, 1, 1, 1],   # river flood / extreme precip  -> ImpactFunc id 1 (RF)
    'impf_TC':       [1, 1, 1, 1],   # tropical cyclone              -> ImpactFunc id 1 (TC)
    'impf_WF':       [1, 1, 1, 1],   # wildfire                      -> ImpactFunc id 1 (WF)
    'impf_LS':       [1, 1, 1, 1],   # landslide                     -> ImpactFunc id 1 (LS)
    'impf_HS':       [1, 1, 1, 1],   # heat stress / heatwave        -> ImpactFunc id 1 (HS)
    'impf_RC':       [1, 1, 1, 1],   # drought / water stress (RC)   -> ImpactFunc id 1 (RC)
    'impf_CF':       [1, 1, 1, 1],   # sea-level rise / coastal flood-> ImpactFunc id 1 (CF)
})

exp = Exposures(sites)
exp.check()
exp.gdf.head()

## 3 · Define the 72-run matrix (scenarios × time horizons × hazards)

**Scenarios.** CLIMADA's open hazard API is indexed by RCP labels. We map each SSP to the
closest available RCP that exists for **both** flood and TC hazards:

| ESRS scenario | ~Warming | River flood | Tropical cyclone | Note |
|---|---|---|---|---|
| SSP1-1.9 | +1.5 °C | `rcp26` | `rcp26` | SSP1-2.6 used as conservative proxy (document the ~0.3 °C) |
| SSP2-4.5 | +2.7 °C | `rcp60` | `rcp60` | `rcp45` is absent from the flood dataset → `rcp60` for both |
| SSP3-7.0 | +3.6 °C | `rcp85` | `rcp85` | High-emission scenario (satisfies E1-2 §16(a)(i)) |

**Time horizons.** River flood uses a `year_range` window; tropical cyclone uses a single
`ref_year` snapshot; hazards without scenario projections use `time_prop=None`.

---

### CLIMADA ETH Data API — availability per hazard

The five new hazards do **not** behave like flood/TC in the CLIMADA API. The table below
explains the exact error each produced in the first run and the fix applied in `hazard_config`:

| Hazard | Error observed | Root cause | Fix in this notebook |
|---|---|---|---|
| Flooding | — (works) | Fully supported | Per-country, `year_range` |
| Extreme Precip. | — (works) | Same dataset as Flooding | Per-country, `year_range` |
| Storms | `NoResult` (rcp85+2080 only) | Dataset not published for that combo | Recorded as NaN |
| **Wildfires** | `NoResult` all combos | API has historical FIRMS catalog only — `climate_scenario` / `year_range` filters don't exist for this type | `time_prop=None`: query per-country without scenario or time filters; same historical baseline reused across all 9 cells |
| **Landslides** | `ValueError` all combos | `'landslide'` is **not a registered `haz_type`** in the CLIMADA ETH API — the schema rejects it before any HTTP request | `api_unavailable=True`: skip API; alternatives below |
| **Heat Stress** | `ValueError` all combos | `'heat_stress'` is **not a registered `haz_type`** in the CLIMADA ETH API | `api_unavailable=True`: skip API; alternatives below |
| **Drought & Water Stress** | `NoResult` all combos | Dataset is **global** (no per-country tiles) — `country_iso3alpha` filter returns nothing | `country_independent=True`: single global download, no country filter |
| **Sea-Level Rise** | `ValueError` all combos | `'coastal_flood'` is **not a registered `haz_type`** in the CLIMADA ETH API | `api_unavailable=True`: skip API; alternatives below |

**Alternative sources for `api_unavailable` hazards** (wire these in when data is available):

| Hazard | Recommended alternative | URL |
|---|---|---|
| Landslides | `climada_petals` `Landslide` class with NASA COOLR catalog | https://gpm.nasa.gov/landslides/index.html |
| Landslides | UNDRR ThinkHazard API | https://thinkhazard.org/ |
| Heat Stress | ISIMIP3b WBGT exceedance-day NetCDF → load as custom `Hazard` | https://data.isimip.org/ |
| Heat Stress | Copernicus CDS heat-stress indicators | https://cds.climate.copernicus.eu/ |
| Sea-Level Rise | IPCC AR6 interactive SLR projection tool | https://sealevel.nasa.gov/ipcc-ar6-sea-level-projection-tool |
| Sea-Level Rise | `climada_petals` `CoastSeg` module | https://github.com/CLIMADA-project/climada_petals |
| Sea-Level Rise | WRI AQUEDUCT Coastal | https://www.wri.org/aqueduct |

In [ ]:
# SSP -> RCP (intersection that exists for BOTH hazards in the open API)
scenarios = {
    'SSP1-1.9': 'rcp26',   # Paris-aligned (SSP1-2.6 proxy)
    'SSP2-4.5': 'rcp60',   # rcp45 not in flood data -> rcp60 for both
    'SSP3-7.0': 'rcp85',   # high emissions
}

# Per-hazard configuration.
#
#   time_prop          : API property for the time dimension; None = no time filtering
#   timeframes         : ESRS horizon label -> API value (None when time_prop is None)
#   extra_props        : additional fixed API filters (crop_type, model_name, etc.)
#   country_independent: True = global dataset; single download without country_iso3alpha
#   api_unavailable    : True = haz_type not registered in CLIMADA ETH API (ValueError)
#   alt_source         : authoritative alternative when api_unavailable=True
hazard_config = {
    'Flooding': {
        'haz_type': 'river_flood', 'tag': 'RF', 'impf_col': 'impf_RF',
        'time_prop': 'year_range',
        'timeframes': {'Short (0-3yr)': '2010_2030',
                       'Medium (3-10yr)': '2030_2050',
                       'Long (10+yr)': '2050_2070'},
        'extra_props': {},
    },
    'Extreme Precip.': {
        'haz_type': 'river_flood', 'tag': 'RF', 'impf_col': 'impf_RF',
        'time_prop': 'year_range',
        'timeframes': {'Short (0-3yr)': '2010_2030',
                       'Medium (3-10yr)': '2030_2050',
                       'Long (10+yr)': '2050_2070'},
        'extra_props': {},
    },
    'Storms': {
        'haz_type': 'tropical_cyclone', 'tag': 'TC', 'impf_col': 'impf_TC',
        'time_prop': 'ref_year',
        'timeframes': {'Short (0-3yr)': '2040',
                       'Medium (3-10yr)': '2060',
                       'Long (10+yr)': '2080'},
        'extra_props': {'model_name': 'random_walk'},
    },
    'Wildfires': {
        # Root cause of previous NoResult: CLIMADA API wildfire catalog is historical
        # FIRMS data only — the API has no climate_scenario or year_range dimension for
        # this haz_type.  Fix: time_prop=None omits those filters; load_hazard queries
        # per-country without scenario/time.  The cache collapses all 9 scenario×timeframe
        # cells to one entry (historical baseline applied uniformly — conservative for E1-2).
        # For scenario projections download ISIMIP3b fire data from data.isimip.org.
        'haz_type': 'wildfire', 'tag': 'WF', 'impf_col': 'impf_WF',
        'time_prop': None,
        'timeframes': {'Short (0-3yr)': None, 'Medium (3-10yr)': None, 'Long (10+yr)': None},
        'extra_props': {},
    },
    'Landslides': {
        # Root cause of ValueError: 'landslide' is not registered in the CLIMADA ETH
        # Data API schema.  The client rejects it before any HTTP request is made.
        # Alternatives: climada_petals Landslide (NASA COOLR) | UNDRR ThinkHazard API
        #               | JRC Global Risk Data Platform
        'haz_type': 'landslide', 'tag': 'LS', 'impf_col': 'impf_LS',
        'time_prop': None,
        'timeframes': {'Short (0-3yr)': None, 'Medium (3-10yr)': None, 'Long (10+yr)': None},
        'extra_props': {},
        'api_unavailable': True,
        'alt_source': 'climada_petals Landslide (NASA COOLR) | UNDRR ThinkHazard (thinkhazard.org) | JRC GRDP',
    },
    'Heat Stress': {
        # Root cause of ValueError: 'heat_stress' is not registered in the CLIMADA ETH
        # Data API schema.
        # Alternatives: ISIMIP3b WBGT NetCDF (data.isimip.org) loaded as custom Hazard
        #               | Copernicus CDS heat-stress indicators (cds.climate.copernicus.eu)
        'haz_type': 'heat_stress', 'tag': 'HS', 'impf_col': 'impf_HS',
        'time_prop': None,
        'timeframes': {'Short (0-3yr)': None, 'Medium (3-10yr)': None, 'Long (10+yr)': None},
        'extra_props': {},
        'api_unavailable': True,
        'alt_source': 'ISIMIP3b WBGT (data.isimip.org) | Copernicus CDS (cds.climate.copernicus.eu)',
    },
    'Drought & Water Stress': {
        # Root cause of previous NoResult: relative_cropyield is a global gridded dataset
        # — there are no per-country tiles.  Querying with country_iso3alpha returns nothing.
        # Fix: country_independent=True causes load_hazard to skip the per-country loop and
        # issue one global download without a country filter.
        # Note: rcp60 may not be available for this dataset; if SSP2-4.5 returns NoResult,
        # check available scenarios with: Client().list_datasets('relative_cropyield')
        'haz_type': 'relative_cropyield', 'tag': 'RC', 'impf_col': 'impf_RC',
        'time_prop': 'year_range',
        'timeframes': {'Short (0-3yr)': '2010_2030',
                       'Medium (3-10yr)': '2030_2050',
                       'Long (10+yr)': '2050_2070'},
        'extra_props': {'crop_type': 'whe', 'irrigation_type': 'rf'},
        'country_independent': True,
    },
    'Sea-Level Rise': {
        # Root cause of ValueError: 'coastal_flood' is not registered in the CLIMADA ETH
        # Data API schema.
        # Alternatives: IPCC AR6 SLR tool (sealevel.nasa.gov/ipcc-ar6-sea-level-projection-tool)
        #               | climada_petals CoastSeg | WRI AQUEDUCT Coastal (wri.org/aqueduct)
        'haz_type': 'coastal_flood', 'tag': 'CF', 'impf_col': 'impf_CF',
        'time_prop': None,
        'timeframes': {'Short (0-3yr)': None, 'Medium (3-10yr)': None, 'Long (10+yr)': None},
        'extra_props': {},
        'api_unavailable': True,
        'alt_source': 'IPCC AR6 SLR tool (sealevel.nasa.gov) | climada_petals CoastSeg | WRI AQUEDUCT Coastal',
    },
}

COUNTRIES = sites['country_iso3'].unique().tolist()
print('Hazards :', list(hazard_config))
print('Scenarios:', list(scenarios))
print('Countries:', COUNTRIES)
print('Total runs:', len(hazard_config) * len(scenarios) * 3)

print('\n--- CLIMADA API status per hazard ---')
for _name, _c in hazard_config.items():
    if _c.get('api_unavailable'):
        status = f'NOT in API (ValueError) → {_c["alt_source"]}'
    elif _c.get('country_independent'):
        status = 'global dataset (country_independent=True)'
    elif _c['time_prop'] is None:
        status = 'historical catalog only (no scenario/year filters)'
    else:
        status = f'fully supported: per-country + {_c["time_prop"]}'
    print(f'  {_name:<28s} {status}')

## 4 · Build the impact (vulnerability) functions

One generic impact function is defined per hazard type. All use `id=1` so every site maps
to the same curve — replace with site-specific IDs if you have differentiated vulnerability
data (e.g., reinforced concrete vs. timber frame for flood).

| Hazard tag | Intensity metric | Curve basis |
|---|---|---|
| `TC` | Max wind speed (m/s) | Emanuel (2011) — field standard |
| `RF` | Flood depth (m) | JRC-style depth-damage |
| `WF` | Fire weather index (FWI) category 0–5 | Generic fire intensity |
| `LS` | Landslide susceptibility index 0–1 | Generic slope-failure damage |
| `HS` | Annual exceedance days above WBGT threshold | Labour/operational disruption |
| `RC` | Fractional crop-yield loss 0–1 (rainfed wheat) | Proportional supply-chain proxy |
| `CF` | Coastal flood / SLR inundation depth (m) | JRC-style depth-damage (same as RF) |

> The hazard tag must match the impact function's `haz_type` **and** the exposure column
> `impf_{TAG}`. The open river-flood datasets use tag **`RF`** (not `FL`), coastal flood
> uses **`CF`**, drought/cropyield uses **`RC`**.

In [ ]:
# Tropical cyclone (Emanuel 2011) — haz_type 'TC', id 1
impf_tc = ImpfTropCyclone.from_emanuel_usa()

# River flood / extreme precip — depth (m) -> damage fraction. haz_type 'RF', id 1
impf_rf = ImpactFunc(
    id=1,
    haz_type='RF',
    name='Flood depth-damage (generic)',
    intensity=np.array([0, 0.5, 1, 1.5, 2, 3, 5, 10]),  # flood depth, metres
    mdd=np.array([0, 0.15, 0.30, 0.45, 0.55, 0.70, 0.85, 1.0]),
    paa=np.ones(8),
    intensity_unit='m',
)

# Wildfire — fire weather index (FWI) category -> damage fraction. haz_type 'WF', id 1
impf_wf = ImpactFunc(
    id=1,
    haz_type='WF',
    name='Wildfire (fire weather index)',
    intensity=np.array([0, 1, 2, 3, 4, 5]),   # FWI category 0 (none) – 5 (extreme)
    mdd=np.array([0.0, 0.05, 0.15, 0.35, 0.65, 1.0]),
    paa=np.ones(6),
    intensity_unit='FWI category',
)

# Landslide — susceptibility index (0–1) -> damage fraction. haz_type 'LS', id 1
impf_ls = ImpactFunc(
    id=1,
    haz_type='LS',
    name='Landslide (susceptibility-damage)',
    intensity=np.array([0, 0.25, 0.50, 0.75, 1.0]),   # 0 = none, 1 = very high susceptibility
    mdd=np.array([0.0, 0.10, 0.30, 0.60, 1.0]),
    paa=np.ones(5),
    intensity_unit='susceptibility index (0–1)',
)

# Heat stress / heatwave — annual exceedance days above WBGT threshold -> disruption fraction
# haz_type 'HS', id 1
impf_hs = ImpactFunc(
    id=1,
    haz_type='HS',
    name='Heat stress (labour/operational disruption)',
    intensity=np.array([0, 5, 10, 20, 30, 50, 100]),   # exceedance days per year
    mdd=np.array([0.0, 0.02, 0.05, 0.12, 0.25, 0.50, 0.80]),
    paa=np.ones(7),
    intensity_unit='days/yr above threshold',
)

# Drought / water stress — fractional crop-yield loss (rainfed wheat proxy) -> impact fraction
# haz_type 'RC' (relative_cropyield), id 1
impf_rc = ImpactFunc(
    id=1,
    haz_type='RC',
    name='Drought / water stress (crop-yield proxy)',
    intensity=np.array([0, 0.10, 0.20, 0.35, 0.50, 0.75, 1.0]),  # fractional yield loss
    mdd=np.array([0.0, 0.08, 0.18, 0.32, 0.50, 0.72, 1.0]),
    paa=np.ones(7),
    intensity_unit='fractional yield loss (0–1)',
)

# Sea-level rise / coastal flood — inundation depth (m) -> damage fraction. haz_type 'CF', id 1
impf_cf = ImpactFunc(
    id=1,
    haz_type='CF',
    name='Coastal flood / SLR (depth-damage)',
    intensity=np.array([0, 0.5, 1, 1.5, 2, 3, 5, 10]),   # inundation depth, metres
    mdd=np.array([0, 0.15, 0.30, 0.45, 0.55, 0.70, 0.85, 1.0]),
    paa=np.ones(8),
    intensity_unit='m',
)

impf_sets = {
    'RF': ImpactFuncSet([impf_rf]),
    'TC': ImpactFuncSet([impf_tc]),
    'WF': ImpactFuncSet([impf_wf]),
    'LS': ImpactFuncSet([impf_ls]),
    'HS': ImpactFuncSet([impf_hs]),
    'RC': ImpactFuncSet([impf_rc]),
    'CF': ImpactFuncSet([impf_cf]),
}
print('Impact functions ready:', {k: v.get_ids() for k, v in impf_sets.items()})

## 5 · Hazard loader (open-access, cached)

`load_hazard` operates in one of three modes, selected automatically from `hazard_config`:

| Mode | Flag | When used | Behaviour |
|---|---|---|---|
| **Per-country** | *(default)* | `river_flood`, `tropical_cyclone`, `wildfire` | One API call per country; results merged with `Hazard.concat` |
| **Global dataset** | `country_independent=True` | `relative_cropyield` (no country tiles) | One API call without `country_iso3alpha`; single global hazard returned |
| **API unavailable** | `api_unavailable=True` | `landslide`, `heat_stress`, `coastal_flood` | Returns `None` immediately — type not registered in CLIMADA ETH API |

Two additional robustness features apply to the per-country and global modes:

- **Graceful gaps.** Missing country/scenario tiles are skipped and logged; affected sites receive NaN rather than crashing the run.
- **Caching.** Each unique `(haz_type, rcp_code, time_value)` triplet is fetched only once.
  For `time_prop=None` hazards (wildfire), the cache key collapses all 9 scenario×timeframe
  cells to a single entry — the historical baseline is reused without re-downloading.

In [ ]:
import time
from climada.util.api_client import Download

client = Client()
_hazard_cache = {}

def load_hazard(cfg, rcp_code, time_value):
    """Return a merged Hazard for all COUNTRIES, or None if no data exists.

    Selects one of three query modes based on hazard_config flags:
      Mode 1 — per-country   : one API call per ISO3 country; results merged with Hazard.concat
      Mode 2 — global dataset : one API call without country_iso3alpha (country_independent=True)
      Mode 3 — API unavailable: haz_type not in CLIMADA ETH schema; returns None immediately
    """
    # Mode 3: haz_type not registered in the CLIMADA ETH API (raises ValueError)
    if cfg.get('api_unavailable', False):
        return None

    # Cache key: hazards with time_prop=None (e.g. wildfire historical) always return
    # the same dataset regardless of scenario/timeframe → collapse to one cache entry
    has_time = (cfg['time_prop'] is not None) and (time_value is not None)
    cache_key = (cfg['haz_type'],
                 rcp_code    if has_time else None,
                 time_value  if has_time else None)
    if cache_key in _hazard_cache:
        return _hazard_cache[cache_key]

    def _build_props(iso3=None):
        props = {**cfg['extra_props']}
        if iso3:
            props['country_iso3alpha'] = iso3
        if has_time:
            props['climate_scenario'] = rcp_code
            props[cfg['time_prop']]   = time_value
        return props

    # ── Mode 2: global dataset (single download) ────────────────────────────────
    if cfg.get('country_independent', False):
        haz = None
        for attempt in range(2):
            try:
                haz = client.get_hazard(cfg['haz_type'], properties=_build_props())
                break
            except Exception as exc:
                exc_name = type(exc).__name__
                if exc_name in ('ChunkedEncodingError', 'Failed') and attempt == 0:
                    print(f'    - {exc_name} for global {cfg["haz_type"]} '
                          f'({rcp_code}, {time_value}): retrying...')
                    time.sleep(5)
                else:
                    print(f"    - no {cfg['haz_type']} global "
                          f"({rcp_code}, {time_value}): {exc_name}")
                    break
        _hazard_cache[cache_key] = haz
        return haz

    # ── Mode 1: per-country (standard) ─────────────────────────────────────────
    parts = []
    for iso3 in COUNTRIES:
        for attempt in range(2):
            try:
                parts.append(client.get_hazard(cfg['haz_type'],
                                               properties=_build_props(iso3)))
                break
            except Exception as exc:
                exc_name = type(exc).__name__
                if exc_name in ('ChunkedEncodingError', 'Failed') and attempt == 0:
                    print(f'    - {exc_name} for {iso3} ({rcp_code}, {time_value}): '
                          f'purging stale cache & retrying...')
                    for _d in list(Download.select()):
                        if _d.enddownload is None and cfg['haz_type'] in _d.path:
                            _d.delete_instance()
                    time.sleep(5)
                else:
                    print(f"    - no {cfg['haz_type']} for {iso3} "
                          f"({rcp_code}, {time_value}): {exc_name}")
                    break

    haz = None if not parts else (parts[0] if len(parts) == 1 else Hazard.concat(parts))
    _hazard_cache[cache_key] = haz
    return haz

## 6 · Run the 27-combination engine

For each (hazard × scenario × horizon) we fetch the open hazard set and compute the impact
with `ImpactCalc`. `imp.eai_exp[i]` is the Expected Annual Loss for site *i* (aligned to the
exposure order). Failed combinations are recorded with `eal = NaN` so the disclosure table
keeps every site/hazard row and the gaps are explicit rather than silently dropped.

In [ ]:
# Purge stale TC cache entries before the run.
# Queries the CLIMADA Download DB directly — catches entries even when no file
# was written to disk (glob-based purge misses these).
from pathlib import Path
from climada.util.api_client import Download

_purged = 0
_tc_haz_type = hazard_config['Storms']['haz_type']
for _d in list(Download.select()):
    if _d.enddownload is None and _tc_haz_type in _d.path:
        _d.delete_instance()
        print(f'Purged: {Path(_d.path).name}')
        _purged += 1

_hazard_cache.clear()
print(f'\nTotal purged: {_purged}. In-memory cache cleared.')


In [ ]:
from pathlib import Path
from datetime import datetime

all_results = []
n_total = len(hazard_config) * len(scenarios) * 3
n = 0
_api_unavailable_announced = set()   # print alt_source only once per hazard type

for haz_name, cfg in hazard_config.items():
    for ssp_name, rcp_code in scenarios.items():
        for tf_name, tf_value in cfg['timeframes'].items():
            n += 1
            print(f"[{n:2d}/{n_total}] {haz_name} | {ssp_name} | {tf_name}")
            eal_by_site = {}
            try:
                if cfg.get('api_unavailable', False):
                    # Print the alternative source once per hazard (not 9×)
                    if haz_name not in _api_unavailable_announced:
                        print(f'    -> not in CLIMADA ETH Data API — '
                              f'use: {cfg["alt_source"]}')
                        _api_unavailable_announced.add(haz_name)
                else:
                    hazard = load_hazard(cfg, rcp_code, tf_value)
                    if hazard is None:
                        print('    -> no data for any site; recording NaN')
                    else:
                        imp = ImpactCalc(exp, impf_sets[cfg['tag']], hazard).impact(
                            save_mat=False, assign_centroids=True)
                        for i, name in enumerate(sites['site_name']):
                            eal_by_site[name] = float(imp.eai_exp[i])
            except Exception as exc:
                print(f'    -> run failed: {type(exc).__name__}: {exc}')

            for _, row in sites.iterrows():
                all_results.append({
                    'site_name':     row['site_name'],
                    'latitude':      row['latitude'],
                    'longitude':     row['longitude'],
                    'country_iso3':  row['country_iso3'],
                    'headcount':     row['headcount'],
                    'business_area': row['business_area'],
                    'criticality':   row['criticality'],
                    'hazard':        haz_name,
                    'scenario':      ssp_name,
                    'timeframe':     tf_name,
                    'eal':           eal_by_site.get(row['site_name'], np.nan),
                })

base_path = Path('simulations')
today = datetime.now().strftime('%Y%m%d')
prefix = f'{today}_SIM_NUM_'
base_path.mkdir(parents=True, exist_ok=True)

existing = [p for p in base_path.iterdir() if p.is_dir() and p.name.startswith(prefix)]
sim_nums = [int(p.name[len(prefix):]) for p in existing if p.name[len(prefix):].isdigit()]
run_dir = base_path / f'{prefix}{max(sim_nums, default=0) + 1}'
run_dir.mkdir()

results_df = pd.DataFrame(all_results)
out_path = run_dir / 'e1_2_climada_results.csv'
results_df.to_csv(out_path, index=False)
print(f'\nDone. {len(results_df)} rows -> {out_path}')
results_df.head(12)

## 6 · Run the 72-combination engine

For each (hazard × scenario × horizon) we fetch the open hazard set and compute the impact
with `ImpactCalc`. `imp.eai_exp[i]` is the Expected Annual Loss for site *i* (aligned to the
exposure order). Failed combinations are recorded with `eal = NaN` so the disclosure table
keeps every site/hazard row and the gaps are explicit rather than silently dropped.

For the five new hazards (wildfire, landslide, heat stress, drought, sea-level rise) the
CLIMADA open API has more limited global coverage than flood/TC. `NaN` entries in those rows
signal that an alternative data source (e.g. local government hazard maps, INFORM Risk Index,
or a specialist model) is required to complete the quantitative assessment — this is
documented in §8.

In [ ]:
if results_df['eal'].notna().any():
    pivot = results_df.pivot_table(
        index=['site_name', 'business_area', 'criticality', 'headcount', 'hazard'],
        columns=['scenario', 'timeframe'],
        values='eal',
        aggfunc='sum',
        dropna=False,
    )

    site_summary = (results_df.groupby('site_name')
        .agg(total_eal=('eal', 'sum'),
             peak_hazard_eal=('eal', 'max'),
             headcount=('headcount', 'first'),
             criticality=('criticality', 'first'))
        .sort_values('total_eal', ascending=False))

    scenario_summary = (results_df.groupby(['scenario', 'timeframe'])
        .agg(total_eal=('eal', 'sum'),
             max_site_eal=('eal', 'max'),
             sites_at_risk=('eal', lambda s: int((s > 0).sum())))
        .reset_index())

    out_xlsx = run_dir / 'e1_2_disclosure_table.xlsx'
    with pd.ExcelWriter(out_xlsx) as xl:
        pivot.to_excel(xl, sheet_name='EAL_by_site_hazard')
        site_summary.to_excel(xl, sheet_name='Site_summary')
        scenario_summary.to_excel(xl, sheet_name='Scenario_summary')

    print(f'Wrote {out_xlsx} (3 sheets)\n')
    print('=== EAL (fraction of asset value) by site & hazard ===')
    display(pivot.round(4))
    print('\n=== Site summary (most exposed first) ===')
    display(site_summary.round(4))
    print('\n=== Scenario summary ===')
    print(scenario_summary.round(4).to_string(index=False))
else:
    print('No EAL values were produced — check network access to the CLIMADA Data API and '
          'that the requested scenario/year combinations exist for your countries.')

Wrote e1_2_disclosure_table.xlsx (3 sheets)

=== EAL (fraction of asset value) by site & hazard ===


scenario                                                             SSP1-1.9  \
timeframe                                                        Long (10+yr)   
site_name    business_area criticality headcount hazard                         
London HQ    Corporate HQ  Critical    85        Extreme Precip.          NaN   
                                                 Flooding                 NaN   
                                                 Storms                   NaN   
                                       120       Extreme Precip.          NaN   
                                                 Flooding                 NaN   
...                                                                       ...   
Singapore DC Sales         Medium      200       Flooding                 NaN   
                                                 Storms                   NaN   
                                       450       Extreme Precip.          NaN   
                                                 Flooding                 NaN   
                                                 Storms                   NaN   

scenario                                                                          \
timeframe                                                        Medium (3-10yr)   
site_name    business_area criticality headcount hazard                            
London HQ    Corporate HQ  Critical    85        Extreme Precip.             NaN   
                                                 Flooding                    NaN   
                                                 Storms                      NaN   
                                       120       Extreme Precip.             NaN   
                                                 Flooding                    NaN   
...                                                                          ...   
Singapore DC Sales         Medium      200       Flooding                    NaN   
                                                 Storms                      NaN   
                                       450       Extreme Precip.             NaN   
                                                 Flooding                    NaN   
                                                 Storms                      NaN   

scenario                                                                        \
timeframe                                                        Short (0-3yr)   
site_name    business_area criticality headcount hazard                          
London HQ    Corporate HQ  Critical    85        Extreme Precip.           NaN   
                                                 Flooding                  NaN   
                                                 Storms                    NaN   
                                       120       Extreme Precip.           NaN   
                                                 Flooding                  NaN   
...                                                                        ...   
Singapore DC Sales         Medium      200       Flooding                  NaN   
                                                 Storms                    NaN   
                                       450       Extreme Precip.           NaN   
                                                 Flooding                  NaN   
                                                 Storms                    NaN   

scenario                                                             SSP2-4.5  \
timeframe                                                        Long (10+yr)   
site_name    business_area criticality headcount hazard                         
London HQ    Corporate HQ  Critical    85        Extreme Precip.          NaN   
                                                 Flooding                 NaN   
                                                 Storms                   NaN   
                                       120       Extr


=== Site summary (most exposed first) ===


,total_eal,peak_hazard_eal,headcount,criticality
site_name,,,,
Manila WH,0.0825,0.0123,120,High
Paris Office,0.0200,0.0025,200,Medium
London HQ,0.0004,0.0001,450,Critical
Singapore DC,0.0000,0.0000,85,Critical



=== Scenario summary ===
scenario       timeframe  total_eal  max_site_eal  sites_at_risk
SSP1-1.9    Long (10+yr)     0.0023        0.0010              5
SSP1-1.9 Medium (3-10yr)     0.0109        0.0098              4
SSP1-1.9   Short (0-3yr)     0.0125        0.0099              6
SSP2-4.5    Long (10+yr)     0.0143        0.0119              6
SSP2-4.5 Medium (3-10yr)     0.0148        0.0108              6
SSP2-4.5   Short (0-3yr)     0.0134        0.0100              6
SSP3-7.0    Long (10+yr)     0.0066        0.0025              4
SSP3-7.0 Medium (3-10yr)     0.0160        0.0123              6
SSP3-7.0   Short (0-3yr)     0.0122        0.0109              6


## 8 · ESRS E1-2 datapoint mapping

How the outputs above satisfy the draft E1-2 requirements:

| Paragraph | Requirement | Satisfied by |
|---|---|---|
| §14 | Classify each risk as physical / transition | `hazard` column — all eight are **physical** risks |
| §15(a) | Methodology for exposure to hazards | CLIMADA (ETH Zurich, peer-reviewed, used by EIOPA) + this notebook |
| §15(b) | Transition events & trends | **Out of scope** — CLIMADA is physical-risk only; needs a separate assessment |
| §16(a)(i) | ≥1 high-emission scenario | **SSP3-7.0** (`rcp85`) |
| §16(a)(iii) | Temperature projections | SSP→°C table in §3 |
| §16(b) | Scope of operations | `site_name`, `latitude`, `longitude`, `country_iso3` |
| §16(c) | Key assumptions | Impact functions (§4), unit asset value, RCP proxies (§3) |
| §16(d) | Time period of analysis | `timeframe` column / Short–Medium–Long horizons |
| AR 6(a) | Screen which assets are exposed | `eal > 0` flags exposed sites |
| AR 6(b) | Sensitivity considering location | `eal` + `headcount`, `business_area`, `criticality` |

**Documented limitations** (state these in the narrative for audit defensibility):

1. *Extreme precipitation* is derived from the river-flood model (precip → flood → damage),
   per the standard CLIMADA approach — EAL mirrors the flood result.
2. *Tropical cyclone* covers only cyclone-prone basins; European sites (London, Paris) show no
   TC risk. Extratropical windstorm risk for Europe needs the Petals `StormEurope` module.
3. SSP2-4.5 uses `rcp60` (flood data lacks `rcp45`) and SSP1-1.9 uses the SSP1-2.6 proxy;
   both substitutions are conservative and should be disclosed.
4. With `value = 1.0`, EAL is a damage **fraction**; multiply by real asset values for monetary
   loss before reporting figures externally.
5. *Wildfires* — CLIMADA ETH API `wildfire` catalog is **historical FIRMS data only**
   (no `climate_scenario` / `year_range` dimension). The notebook applies the historical
   fire-frequency baseline uniformly across all scenario×timeframe cells (conservative).
   For scenario projections, download ISIMIP3b fire data from `data.isimip.org` and load
   it as a custom `Hazard` object.
6. *Landslides* — `'landslide'` is **not a registered `haz_type`** in the CLIMADA ETH Data
   API (raises `ValueError`). All 9 combinations record `NaN`. Recommended path:
   `climada_petals` `Landslide` class with the NASA COOLR catalog, or the UNDRR ThinkHazard
   API (`thinkhazard.org`).
7. *Heat Stress / Heatwaves* — `'heat_stress'` is **not a registered `haz_type`** in the
   CLIMADA ETH Data API (raises `ValueError`). All 9 combinations record `NaN`. Recommended
   path: download ISIMIP3b WBGT exceedance-day NetCDF from `data.isimip.org` and load as a
   custom `Hazard` object, or use Copernicus CDS heat-stress indicators
   (`cds.climate.copernicus.eu`). The impact function (`impf_hs`) is already defined —
   only the hazard data source needs to be wired in.
8. *Drought & Water Stress* — `relative_cropyield` **is** in the CLIMADA API but only as a
   global gridded dataset; per-country queries (`country_iso3alpha`) return `NoResult`.
   Fixed via `country_independent=True` (single global download). If SSP2-4.5 (`rcp60`)
   still returns `NoResult`, verify available scenario labels with
   `Client().list_datasets('relative_cropyield')`.
9. *Sea-Level Rise* — `'coastal_flood'` is **not a registered `haz_type`** in the CLIMADA
   ETH Data API (raises `ValueError`). All 9 combinations record `NaN`. Recommended paths:
   IPCC AR6 interactive SLR tool (`sealevel.nasa.gov/ipcc-ar6-sea-level-projection-tool`),
   `climada_petals` `CoastSeg`, or WRI AQUEDUCT Coastal (`wri.org/aqueduct`). The impact
   function (`impf_cf`) is already defined — only the hazard data source needs wiring in.